# Quantum Simulation of Allosteric Signal Propagation

*2026 Global Quantum + AI Challenge*

70-80% of known disease-relevant proteins are considered "undruggable" — their active
sites resist direct binding by small molecules. **Allosteric drugs** offer an alternative:
binding at a distant site to modulate protein function through conformational signal propagation.

Classical molecular dynamics struggles to capture these rare, long-range conformational
events. This notebook demonstrates how uniqx simulates allosteric signal propagation
using quantum Hamiltonian dynamics on hybrid hardware:

| Workload | Operation | Hardware affinity |
|:---------|:----------|:-----------------:|
| Residue coupling Hamiltonian | Kronecker products + embedding (`kron`) | **CPU** |
| Conformational time evolution | Matrix exponential action (`expv`) | **GPU / QPU** |
| Binding energy landscape | Eigendecomposition (`eigs`) | **QPU** |
| Signal propagation measurement | Expectation values (`expect`) | **CPU** |

**Challenge**: Can quantum simulation unlock undruggable targets by revealing allosteric pathways?

In [ ]:
import os
from uniqx import connect

# Default to a local service for development.
# For prod, export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
client = connect(endpoint)
print("Connected to", endpoint)
import math
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx import ops, tracing, parse_result
from uniqx.ops import SX, SY, SZ, I2
from uniqx.core.execution import connect, preflight, submit, get, fmt_mat, fmt_vec, fmt_scalar

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Protein Residue Network Model

We model the protein as a chain of $N$ coupled quantum sites (residues). Each residue
has 2 conformational states: **native** ($|0\rangle$) and **perturbed** ($|1\rangle$).
The full Hilbert space has dimension $2^N$.

The Hamiltonian encodes:
- **Nearest-neighbour backbone couplings** ($XX + YY + ZZ$): conformational crosstalk along the peptide chain
- **Long-range allosteric coupling** between the allosteric site (residue 0) and the active site (residue $N-1$)
- **On-site conformational energies** ($Z_i$): the energy barrier for each residue to switch states

$$H_{\text{protein}} = J_{\text{bb}} \sum_{\langle i,j \rangle} (X_i X_j + Y_i Y_j + Z_i Z_j) + J_{\text{allo}} (X_0 X_{N-1} + Y_0 Y_{N-1} + Z_0 Z_{N-1}) + \sum_i \epsilon_i Z_i$$

where $J_{\text{bb}}$ is the backbone coupling, $J_{\text{allo}}$ is the allosteric
coupling strength, and $\epsilon_i$ are on-site conformational energies.

In [ ]:
# --- Pure-Python matrix helpers (no numpy in traced code) ---


def kron(A, B):
    """Kronecker product of two matrices (lists of lists)."""
    rA, rB = len(A), len(B)
    n = rA * rB
    C = [[0.0] * n for _ in range(n)]
    for i in range(rA):
        for j in range(rA):
            for k in range(rB):
                for l in range(rB):
                    C[i * rB + k][j * rB + l] = A[i][j] * B[k][l]
    return C


def eye(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def matadd(A, B):
    return [[A[i][j] + B[i][j] for j in range(len(A))] for i in range(len(A))]


def matscale(s, A):
    return [[s * x for x in row] for row in A]


def embed(op, site, n_sites):
    """Embed a single-site operator into the full Hilbert space."""
    result = eye(1)
    for k in range(n_sites):
        result = kron(result, op if k == site else I2)
    return result


def two_body(opA, opB, sa, sb, n_sites):
    """Two-site coupling term in the full Hilbert space."""
    parts = []
    for k in range(n_sites):
        if k == sa:
            parts.append(opA)
        elif k == sb:
            parts.append(opB)
        else:
            parts.append(I2)
    result = eye(1)
    for p in parts:
        result = kron(result, p)
    return result


def build_protein_hamiltonian(n_residues, J_backbone, J_allosteric, epsilon):
    """Build the protein residue network Hamiltonian.

    H = J_bb * sum_{<i,j>} (XX + YY + ZZ)       [backbone]
      + J_allo * (X_0 X_{N-1} + Y_0 Y_{N-1} + Z_0 Z_{N-1})  [allosteric]
      + sum_i epsilon_i * Z_i                     [on-site]
    """
    dim = 2 ** n_residues
    H = [[0.0] * dim for _ in range(dim)]

    # Nearest-neighbour backbone couplings (Heisenberg XXX)
    for i in range(n_residues - 1):
        H = matadd(H, matscale(J_backbone, two_body(SX, SX, i, i + 1, n_residues)))
        H = matadd(H, matscale(J_backbone, two_body(SY, SY, i, i + 1, n_residues)))
        H = matadd(H, matscale(J_backbone, two_body(SZ, SZ, i, i + 1, n_residues)))

    # Long-range allosteric coupling (site 0 <-> site N-1)
    if n_residues > 2:
        last = n_residues - 1
        H = matadd(H, matscale(J_allosteric, two_body(SX, SX, 0, last, n_residues)))
        H = matadd(H, matscale(J_allosteric, two_body(SY, SY, 0, last, n_residues)))
        H = matadd(H, matscale(J_allosteric, two_body(SZ, SZ, 0, last, n_residues)))

    # On-site conformational energies
    for i in range(n_residues):
        H = matadd(H, matscale(epsilon[i], embed(SZ, i, n_residues)))

    return H, dim


def build_active_site_observable(n_residues):
    """Build the observable that measures perturbation at the active site.

    M_active = |1><1| on site N-1 = (I - Z_{N-1}) / 2
    Expectation value = probability of active site being in perturbed state.
    """
    dim = 2 ** n_residues
    last = n_residues - 1
    # |1><1| = (I - Z) / 2
    M = matadd(
        matscale(0.5, eye(dim)),
        matscale(-0.5, embed(SZ, last, n_residues)),
    )
    return M


# --- Define protein models ---
PROTEINS = {
    "small_peptide": {
        "n_residues": 4,
        "J_backbone": 1.0,
        "J_allosteric": 0.3,
        "epsilon": [0.2, -0.1, 0.05, -0.15],
        "description": "4-residue peptide (allosteric toy model)",
    },
    "medium_domain": {
        "n_residues": 5,
        "J_backbone": 1.0,
        "J_allosteric": 0.25,
        "epsilon": [0.2, -0.1, 0.05, -0.08, -0.15],
        "description": "5-residue protein domain",
    },
    "kinase_pocket": {
        "n_residues": 6,
        "J_backbone": 1.0,
        "J_allosteric": 0.2,
        "epsilon": [0.2, -0.1, 0.05, -0.08, 0.03, -0.15],
        "description": "6-residue kinase binding pocket",
    },
}

# Build and verify each model
protein_hamiltonians = {}
for name, spec in PROTEINS.items():
    H, dim = build_protein_hamiltonian(
        spec["n_residues"], spec["J_backbone"], spec["J_allosteric"], spec["epsilon"]
    )
    protein_hamiltonians[name] = {"H": H, "dim": dim, **spec}

    # Quick numpy verification
    H_np = np.array(H)
    evals = np.linalg.eigvalsh(H_np)
    print(f"{name} ({spec['description']}):")
    print(f"  dim = {dim}, E_0 = {evals[0]:.6f}, E_1 = {evals[1]:.6f}, gap = {evals[1] - evals[0]:.6f}")
    print(f"  J_bb = {spec['J_backbone']}, J_allo = {spec['J_allosteric']}")
    print()

## 2. Allosteric Signal Propagation

Starting from a **ligand-bound state** at the allosteric site, we evolve the system
under the protein Hamiltonian and measure how the signal propagates to the active site.

The initial state is $|1, 0, 0, \ldots, 0\rangle$: the allosteric site (residue 0)
is perturbed (ligand bound), all other residues are in their native conformation.

We track the **active site perturbation probability**:

$$P_{\text{active}}(t) = \langle \psi(t) | M_{\text{active}} | \psi(t) \rangle$$

where $|\psi(t)\rangle = e^{-iHt} |\psi_0\rangle$ and $M_{\text{active}} = |1\rangle\langle 1|$
on the active site.

In [ ]:
# --- Build initial state: allosteric site excited, others in ground ---
def build_ligand_bound_state(n_residues):
    """Build |1, 0, 0, ..., 0>: allosteric site perturbed, rest native.

    In the computational basis, |1> is index 1 for the first qubit.
    For n qubits, |1,0,...,0> = state index 2^(n-1).
    """
    dim = 2 ** n_residues
    psi = [0.0] * dim
    psi[2 ** (n_residues - 1)] = 1.0  # |1,0,...,0>
    return psi


# --- Trace the allosteric propagation module ---
@tracing.to_module(name="allosteric_propagation")
def allosteric_propagation(H_protein, M_active, psi0, t):
    """Evolve the ligand-bound state and measure active site perturbation.

    Returns:
        psi_t: evolved state
        signal: active site perturbation probability
        energy: total energy (conservation check)
    """
    psi_t = ops.expv(H_protein, psi0, t, hermitian=True)
    signal = ops.expect(M_active, psi_t)
    energy = ops.expect(H_protein, psi_t)
    return psi_t, signal, energy


# Test trace with the small peptide model
spec = PROTEINS["small_peptide"]
H_small = protein_hamiltonians["small_peptide"]["H"]
dim_small = protein_hamiltonians["small_peptide"]["dim"]
M_active_small = build_active_site_observable(spec["n_residues"])
psi0_small = build_ligand_bound_state(spec["n_residues"])

mod_test = allosteric_propagation(H_small, M_active_small, psi0_small, 1.0)
print(f"Allosteric propagation module:")
print(f"  IR size: {len(mod_test.to_text())} chars")
print(f"  System: {spec['n_residues']} residues, dim={dim_small}")
print(f"  Initial state: |1,0,...,0> (ligand bound at allosteric site)")

In [ ]:
# --- Preflight: hardware assignment ---
def print_options(label, options):
    print(f"\n--- {label} ---")
    if not options:
        print("  (no options)")
        return
    print(f"  {'Label':<24} {'Time':>10} {'Cost ($)':>12} {'Error':>8} {'Carbon (g)':>11}")
    print(f"  {'-' * 24} {'-' * 10} {'-' * 12} {'-' * 8} {'-' * 11}")
    for opt in options:
        flag = "  <-- rec" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<24} {opt['total_time']:>10.2f}"
            f"  ${opt['total_cost_usd']:>10.6f}"
            f"  {opt['max_error_rate']:>8.4f}"
            f"  {opt['total_carbon_g']:>10.3f}{flag}"
        )


opts_propagation = preflight(mod_test, client=client)
print_options(f"Allosteric Propagation ({spec['n_residues']} residues, dim={dim_small})", opts_propagation)

## 3. Execute Time Evolution

We sweep over time from $t = 0.1$ to $t = 3.0$ (in units of $1/J_{\text{bb}}$)
and track the allosteric signal arriving at the active site.

A rising $P_{\text{active}}(t)$ indicates that the ligand-induced conformational
change has propagated through the backbone to the active site. The **signal
arrival time** is the first peak — a key pharmacological parameter.

In [ ]:
t_values = np.linspace(0.1, 3.0, 15)

# Pick up to 2 hardware options to compare
options_to_try = []
if opts_propagation.recommended:
    options_to_try.append(opts_propagation.recommended)
for opt in opts_propagation:
    if len(options_to_try) >= 2:
        break
    if opt["label"] != options_to_try[0]["label"]:
        options_to_try.append(opt)

print(f"Comparing hardware: {[o['label'] for o in options_to_try]}")
print(f"Time sweep: t = {t_values[0]:.1f} to {t_values[-1]:.1f} ({len(t_values)} points)")
print(f"System: {spec['n_residues']} residues, dim={dim_small}\n")

sweep_results = {}

for opt in options_to_try:
    label = opt["label"]
    option_idx = opt["_idx"]
    print(f"=== {label} ===")

    sample_t = []
    signals = []
    energies = []
    wall_times = []
    skipped = 0

    for i, t_val in enumerate(t_values):
        mod = allosteric_propagation(
            H_small, M_active_small, psi0_small, float(t_val)
        )

        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=[
                fmt_mat(H_small, dim_small, dim_small),
                fmt_mat(M_active_small, dim_small, dim_small),
                fmt_vec(psi0_small, dim_small),
                fmt_scalar(float(t_val)),
            ],
            preflight_job_id=opts_propagation.job_id,
            option_idx=option_idx,
        )
        res = get(jid, client=client)
        dt = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()

        # Some primitives may currently be unavailable for the selected
        # hardware option. Skip points where the result payload is missing
        # the expected outputs and keep the sweep going.
        out = parse_result(payload, ["psi_t", "signal", "energy"]) if payload else {}
        if "signal" not in out or "energy" not in out:
            skipped += 1
            if skipped <= 2:
                print(f"  t={t_val:.2f}: skipped (no result returned)")
            continue

        sig = out["signal"][2][0]
        E = out["energy"][2][0]

        sample_t.append(float(t_val))
        signals.append(sig)
        energies.append(E)
        wall_times.append(dt)

        if (i + 1) % 5 == 0:
            print(f"  t={t_val:.2f}: P_active={sig:.6f}, E={E:.6f} ({dt:.2f}s)")

    if not signals:
        print(
            f"  Known limitation: time evolution returned no usable points on"
            f" '{label}'. Skipping this option and continuing.\n"
        )
        continue

    sweep_results[label] = {
        "t_values": np.array(sample_t),
        "signals": np.array(signals),
        "energies": np.array(energies),
        "wall_times": np.array(wall_times),
        "option": opt,
        "skipped": skipped,
    }
    if skipped:
        print(f"  Skipped {skipped} of {len(t_values)} points.")
    print(f"  Total: {sum(wall_times):.2f}s\n")

if not sweep_results:
    print(
        "Note: time-evolution sweep produced no points for any hardware option."
        " The plot and summary cells below will adapt accordingly."
    )

## 4. Binding Energy Landscape

The energy spectrum of the protein Hamiltonian reveals binding thermodynamics.
By comparing eigenvalues with and without the allosteric ligand, we compute:

$$\Delta E_{\text{bind}} = E_0^{\text{bound}} - E_0^{\text{unbound}}$$

A **negative** $\Delta E_{\text{bind}}$ indicates favourable ligand binding.
The energy gap $\Delta = E_1 - E_0$ determines the thermal stability of the
allosteric signal at physiological temperature.

In [ ]:
@tracing.to_module(name="binding_energy_spectrum")
def binding_energy(H_mat):
    """Compute the 4 lowest eigenvalues of the protein Hamiltonian."""
    eigenvalues, eigenvectors = ops.eigs(H_mat, k=4, hermitian=True, which="smallest")
    return eigenvalues, eigenvectors


# --- Ligand-bound Hamiltonian: stronger allosteric coupling ---
def build_bound_hamiltonian(name):
    """Build Hamiltonian with enhanced allosteric coupling (ligand effect)."""
    spec = PROTEINS[name]
    # Ligand binding enhances the allosteric coupling by 2x
    # and shifts the allosteric site energy
    epsilon_bound = list(spec["epsilon"])
    epsilon_bound[0] -= 0.5  # ligand stabilises allosteric site
    H_bound, dim = build_protein_hamiltonian(
        spec["n_residues"],
        spec["J_backbone"],
        spec["J_allosteric"] * 2.0,  # enhanced coupling
        epsilon_bound,
    )
    return H_bound, dim


# Compute spectra for bound and unbound states
binding_results = {}

for state_label, H_matrix in [
    ("unbound", H_small),
    ("bound", build_bound_hamiltonian("small_peptide")[0]),
]:
    mod_eigs = binding_energy(H_matrix)
    opts_eigs = preflight(mod_eigs, client=client)
    print_options(f"Eigendecomposition ({state_label})", opts_eigs)

    rec = opts_eigs.recommended
    t0 = time.monotonic()
    jid = submit(
        mod_eigs,
        client=client,
        runtime_inputs=[fmt_mat(H_matrix, dim_small, dim_small)],
        preflight_job_id=opts_eigs.job_id,
        option_idx=rec["_idx"],
    )
    res = get(jid, client=client)
    wall = time.monotonic() - t0

    payload = res.get("payload", b"")
    if isinstance(payload, str):
        payload = payload.encode()
    out = parse_result(payload, ["eigenvalues", "eigenvectors"]) if payload else {}
    if "eigenvalues" not in out:
        print(f"  Known limitation: eigendecomposition returned no result for '{state_label}'. Skipping.")
        continue
    evals = out["eigenvalues"][2]

    binding_results[state_label] = {
        "eigenvalues": evals,
        "time": wall,
        "option": rec,
    }
    print(f"  {state_label}: E = {[f'{e:.4f}' for e in evals]} ({wall:.2f}s)")

# Compute binding energy (only if both states succeeded)
if "bound" in binding_results and "unbound" in binding_results:
    E0_bound = binding_results["bound"]["eigenvalues"][0]
    E0_unbound = binding_results["unbound"]["eigenvalues"][0]
    dE_bind = E0_bound - E0_unbound

    gap_bound = binding_results["bound"]["eigenvalues"][1] - E0_bound
    gap_unbound = binding_results["unbound"]["eigenvalues"][1] - E0_unbound

    print(f"\nBinding energy: dE = {dE_bind:.6f} ({'favourable' if dE_bind < 0 else 'unfavourable'})")
    print(f"Energy gap (bound):   {gap_bound:.6f}")
    print(f"Energy gap (unbound): {gap_unbound:.6f}")
else:
    E0_bound = E0_unbound = dE_bind = float("nan")
    gap_bound = gap_unbound = float("nan")
    print("\nBinding energy: skipped (eigendecomposition unavailable for one or both states).")

## 5. Protein Size Scaling

The computational cost of allosteric simulation scales exponentially with the
number of residues ($2^N$ Hilbert space). We preflight all three protein models
to see how uniqx's cost model adapts hardware assignments as the system
grows.

In [ ]:
scaling_data = []

for name in ["small_peptide", "medium_domain", "kinase_pocket"]:
    spec = PROTEINS[name]
    n = spec["n_residues"]
    H, dim = protein_hamiltonians[name]["H"], protein_hamiltonians[name]["dim"]
    M_active = build_active_site_observable(n)
    psi0 = build_ligand_bound_state(n)

    # Trace propagation module
    mod = allosteric_propagation(H, M_active, psi0, 1.0)
    opts = preflight(mod, client=client)

    print(f"\n{name} (n={n}, dim={dim}):")
    for opt in opts:
        flag = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}"
            f"  cost=${opt['total_cost_usd']:.4f}"
            f"  err={opt['max_error_rate']:.4f}{flag}"
        )
        scaling_data.append({
            "name": name,
            "n": n,
            "dim": dim,
            "label": opt["label"],
            "time": opt["total_time"],
            "cost": opt["total_cost_usd"],
            "error": opt["max_error_rate"],
            "carbon": opt["total_carbon_g"],
            "recommended": opt.get("recommended", False),
        })

    # Also preflight the eigendecomposition module
    mod_eigs = binding_energy(H)
    opts_eigs = preflight(mod_eigs, client=client)
    print(f"  eigs:")
    for opt in opts_eigs:
        flag = " *" if opt.get("recommended") else ""
        print(
            f"    {opt['label']:<20} time={opt['total_time']:>8.1f}"
            f"  cost=${opt['total_cost_usd']:.4f}{flag}"
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# --- Top-left: Allosteric signal propagation curves ---
if sweep_results:
    for label, data in sweep_results.items():
        c = colors_hw.get(label, "#6b7280")
        ts = data.get("t_values", t_values)
        axes[0, 0].plot(
            ts, data["signals"],
            "o-", color=c, label=label, markersize=4,
        )
    axes[0, 0].legend(fontsize=8)
else:
    axes[0, 0].text(
        0.5, 0.5, "Time evolution unavailable\n(sweep produced no points)",
        ha="center", va="center", transform=axes[0, 0].transAxes,
        fontsize=11, color="#6b7280",
    )
axes[0, 0].set_xlabel("Time $t$ (units of $1/J_{bb}$)", fontsize=11)
axes[0, 0].set_ylabel("$P_{\\mathrm{active}}(t)$", fontsize=11)
axes[0, 0].set_title("Allosteric Signal at Active Site")
axes[0, 0].grid(alpha=0.3)
axes[0, 0].axhline(0.5, color="gray", linestyle=":", alpha=0.4, label="_nolegend_")

# --- Top-right: Energy spectrum (bound vs unbound) ---
if "bound" in binding_results and "unbound" in binding_results:
    x_bound = range(len(binding_results["bound"]["eigenvalues"]))
    x_unbound = range(len(binding_results["unbound"]["eigenvalues"]))
    axes[0, 1].bar(
        [x - 0.15 for x in x_unbound],
        binding_results["unbound"]["eigenvalues"],
        width=0.3, color="#2563eb", alpha=0.8, label="Unbound",
    )
    axes[0, 1].bar(
        [x + 0.15 for x in x_bound],
        binding_results["bound"]["eigenvalues"],
        width=0.3, color="#dc2626", alpha=0.8, label="Ligand-bound",
    )
    axes[0, 1].set_title(f"Energy Spectrum ($\\Delta E_{{bind}}$ = {dE_bind:.4f})")
    axes[0, 1].legend(fontsize=9)
else:
    axes[0, 1].text(
        0.5, 0.5, "Energy spectrum unavailable",
        ha="center", va="center", transform=axes[0, 1].transAxes,
        fontsize=11, color="#6b7280",
    )
    axes[0, 1].set_title("Energy Spectrum")
axes[0, 1].set_xlabel("Eigenvalue index", fontsize=11)
axes[0, 1].set_ylabel("Energy", fontsize=11)
axes[0, 1].grid(axis="y", alpha=0.3)

# --- Bottom-left: Time scaling by protein size ---
hw_labels = sorted(set(d["label"] for d in scaling_data))
for hw in hw_labels:
    subset = [d for d in scaling_data if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].semilogy(
        [d["n"] for d in subset],
        [d["time"] for d in subset],
        "o-", color=c, label=hw,
    )
# Mark recommended
rec_pts = [d for d in scaling_data if d["recommended"]]
if rec_pts:
    axes[1, 0].scatter(
        [d["n"] for d in rec_pts],
        [d["time"] for d in rec_pts],
        s=200, facecolors="none", edgecolors="black",
        linewidths=2, zorder=10, label="recommended",
    )
axes[1, 0].set_xlabel("Number of residues", fontsize=11)
axes[1, 0].set_ylabel("Estimated time (log)", fontsize=11)
axes[1, 0].set_title("Time Scaling by Protein Size")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# --- Bottom-right: Cost scaling by protein size ---
for hw in hw_labels:
    subset = [d for d in scaling_data if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].semilogy(
        [d["n"] for d in subset],
        [d["cost"] for d in subset],
        "o-", color=c, label=hw,
    )
if rec_pts:
    axes[1, 1].scatter(
        [d["n"] for d in rec_pts],
        [d["cost"] for d in rec_pts],
        s=200, facecolors="none", edgecolors="black",
        linewidths=2, zorder=10, label="recommended",
    )
axes[1, 1].set_xlabel("Number of residues", fontsize=11)
axes[1, 1].set_ylabel("Cost USD (log)", fontsize=11)
axes[1, 1].set_title("Cost Scaling by Protein Size")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle(
    "Allosteric Signal Propagation: Quantum Simulation on Hybrid Hardware",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()

## 6. Results Summary

### Signal Propagation

In [ ]:
print("Allosteric Signal Propagation Results")
print("=" * 75)

if sweep_results:
    for label, data in sweep_results.items():
        signals = data["signals"]
        energies = data["energies"]
        ts = data.get("t_values", t_values)

        # Signal arrival: first time P_active exceeds 0.1
        arrival_idx = next((i for i, s in enumerate(signals) if s > 0.1), None)
        arrival_t = ts[arrival_idx] if arrival_idx is not None else float("inf")

        # Peak signal
        peak_idx = int(np.argmax(signals))
        peak_t = ts[peak_idx]
        peak_val = signals[peak_idx]

        print(f"\n  {label}:")
        print(f"    Total wall time:       {data['wall_times'].sum():.2f}s")
        print(f"    Mean time/point:       {data['wall_times'].mean():.2f}s")
        print(f"    Signal arrival (>0.1): t = {arrival_t:.2f}")
        print(f"    Peak signal:           P = {peak_val:.6f} at t = {peak_t:.2f}")
        print(f"    Energy conservation:   E_min={energies.min():.6f}, E_max={energies.max():.6f}")
        print(f"    Est. cost/point:       ${data['option']['total_cost_usd']:.6f}")
        print(f"    Est. carbon/point:     {data['option']['total_carbon_g']:.3f}g CO2")
        if data.get("skipped"):
            print(f"    Points skipped:        {data['skipped']}")
else:
    print("  (time evolution sweep produced no usable points)")

print(f"\n\nBinding Energy Analysis")
print("=" * 75)
if "bound" in binding_results and "unbound" in binding_results:
    print(f"  E_0 (unbound):  {E0_unbound:.6f}")
    print(f"  E_0 (bound):    {E0_bound:.6f}")
    print(f"  dE_bind:        {dE_bind:.6f} ({'favourable' if dE_bind < 0 else 'unfavourable'})")
    print(f"  Gap (unbound):  {gap_unbound:.6f}")
    print(f"  Gap (bound):    {gap_bound:.6f}")
else:
    print("  (eigendecomposition unavailable for one or both states)")

print(f"\n\nScaling Summary")
print("=" * 75)
print(f"  {'Model':<18} {'Residues':>8} {'Dim':>6} {'Rec. HW':<20} {'Time':>10} {'Cost':>12}")
print(f"  {'-' * 18} {'-' * 8} {'-' * 6} {'-' * 20} {'-' * 10} {'-' * 12}")
for name in ["small_peptide", "medium_domain", "kinase_pocket"]:
    rec = next((d for d in scaling_data if d["name"] == name and d["recommended"]), None)
    if rec:
        print(
            f"  {name:<18} {rec['n']:>8} {rec['dim']:>6} {rec['label']:<20}"
            f" {rec['time']:>10.2f} ${rec['cost']:>10.6f}"
        )

## Summary

| Aspect | Detail |
|:-------|:-------|
| **Target** | Allosteric signal propagation in protein residue networks |
| **Model** | Heisenberg-coupled chain with long-range allosteric term |
| **Proteins** | 4-residue peptide, 5-residue domain, 6-residue kinase pocket |
| **Primitives** | `expv` (time evolution), `eigs` (binding energy), `expect` (signal measurement) |
| **uniqx routing** | Automatic CPU/GPU/QPU selection via preflight cost model |

### Key Findings

1. **Signal propagation is observable**: The allosteric perturbation at residue 0
 propagates through the backbone to the active site (residue $N-1$) on a timescale
 set by the backbone coupling $J_{\text{bb}}$.

2. **Ligand binding shifts the energy landscape**: The bound-state Hamiltonian has
 a lower ground state energy and modified spectral gap, confirming that allosteric
 ligand binding creates a thermodynamically distinct conformational state.

3. **Hardware scaling**: For small peptides (4 residues, dim=16), CPU is optimal.
 As the system grows to 6+ residues (dim=64+), GPU and QPU options become
 cost-competitive. uniqx handles this routing transparently.

4. **Pharmacological relevance**: The signal arrival time and peak amplitude are
 quantitative descriptors for allosteric drug design. Faster signal propagation
 and higher peak amplitude indicate stronger allosteric coupling — key parameters
 for targeting undruggable proteins.

### Clinical Implications

This quantum simulation approach enables systematic screening of allosteric sites
across protein families. By computing signal propagation and binding energies for
candidate drug molecules, researchers can prioritise allosteric targets before
expensive experimental validation — accelerating the development of therapies
for currently undruggable diseases.